# Level 3 A111 with FinBERT top-layer fine-tuning

Same fusion as Level 3 **A111** (`controlled_fusion.py`), but the article encoder is FinBERT with its top `K` layers trained, instead of frozen Word2Vec.

- Slots: market (14), mean of articles, **task-query attention**, **market→article cross-attention**, availability (2).
- **Gate:** one scalar `sigmoid(affine([market, log1p n_berita]))` multiplies the text slots.
- Head: `14 + 3x768 + 2 -> 64 -> 32 -> 1`, as in Level 3.
- `K = 0` is A111 with frozen FinBERT (control); `K = 2` is the fine-tuned variant.
- Same data and 3 expanding folds. **Test is never loaded** until the final cell.

Deviations from Level 3 (needed because the encoder now trains): article embeddings pass through a non-affine LayerNorm instead of a train-fitted scaler; dropout 0.3 in the head; epoch chosen by validation log loss (not MCC); AdamW with a lower LR on encoder layers; 128 tokens, at most 16 articles per day.

Level 3 reference (Word2Vec, frozen): A111 validation MCC -0.0016, log loss 0.6937; test MCC 0.0052, log loss 0.6859.

Kaggle: upload `train.csv`, `val.csv` (and `test.csv` for the last cell), turn on Internet and GPU. Jobs (K x seed) run in parallel, one per GPU; runs are not bit-reproducible.

In [ ]:
import os, json, time, warnings, threading, queue
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, matthews_corrcoef, roc_auc_score, log_loss
from transformers import AutoTokenizer, AutoModel
warnings.filterwarnings("ignore")

DATA_DIR = Path(os.environ.get("DATA_DIR", "/kaggle/input/usdidr-final"))
if not (DATA_DIR / "train.csv").exists():
    DATA_DIR = Path("data/final")
OUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("a111_output")
OUT_DIR.mkdir(exist_ok=True, parents=True)

ENCODER = os.environ.get("ENCODER", "ProsusAI/finbert")
UNFREEZE = [0, 2]            # top layers to train (0 = frozen control)
SEEDS = [42, 123, 2026]
EPOCHS = 4
MAX_LEN, MAX_ART = 128, 16
BATCH_DAYS = 8
LR_ENC, LR_HEAD, WEIGHT_DECAY, DROPOUT = 2e-5, 1e-3, 0.01, 0.3
QUICK_TEST_DAYS = None       # e.g. 80 for a smoke test
FLAGS = (1, 1, 1)            # task query, cross-attention, gate = A111

MARKET = ["log_return_t","return_lag1","return_lag2","return_lag3","return_lag5","volatility_5","volatility_20",
          "ma_ratio_5","ma_ratio_20","momentum_5","momentum_10","rsi_14","input_gap_days","decision_weekday"]
device = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = max(torch.cuda.device_count(), 1)
LOCK = threading.Lock()
GPU_POOL = queue.Queue()
for i in range(N_GPU):
    GPU_POOL.put(f"cuda:{i}" if device == "cuda" else "cpu")
print(device, N_GPU, "GPU(s)", "| data:", DATA_DIR)

In [ ]:
def load(names):
    f = pd.concat([pd.read_csv(DATA_DIR / n, parse_dates=["Date"]) for n in names]).sort_values("Date").reset_index(drop=True)
    f["articles"] = f.articles.map(json.loads)
    f["fold_roles"] = f.fold_roles.map(json.loads)
    f["log1p_n_berita"] = np.log1p(f.n_berita)
    assert f.Date.is_unique
    return f

dev = load(["train.csv", "val.csv"])
print(len(dev), "development days,", sum(map(len, dev.articles)), "articles")

tok = AutoTokenizer.from_pretrained(ENCODER)
rng0 = np.random.RandomState(0)
enc_ids, day_articles = {}, []

def register(frame):
    new = {a["article_id"]: a["text_transformer"] for w in frame.articles for a in w if a["article_id"] not in enc_ids}
    ids = sorted(new)
    enc_ids.update(dict(zip(ids, tok([new[i] for i in ids], truncation=True, max_length=MAX_LEN)["input_ids"])))
    for w in frame.articles:
        a = sorted(x["article_id"] for x in w)
        if len(a) > MAX_ART:
            a = sorted(rng0.choice(a, MAX_ART, replace=False))
        day_articles.append(a)

register(dev)

def make_X(frame, fit_rows):
    # 14 standardized market features + 2 availability features (scaled, not centered), fit on training rows only
    m = StandardScaler().fit(frame.loc[fit_rows, MARKET])
    a = StandardScaler(with_mean=False).fit(frame.loc[fit_rows, ["log1p_n_berita", "has_news"]])
    return np.hstack([m.transform(frame[MARKET]), a.transform(frame[["log1p_n_berita", "has_news"]])]).astype(np.float32)

In [ ]:
class A111(nn.Module):
    def __init__(self, k, flags=FLAGS, d=768):
        super().__init__()
        self.flags, self.d = flags, d
        self.enc = AutoModel.from_pretrained(ENCODER)
        for p in self.enc.parameters():
            p.requires_grad = False
        layers = list(self.enc.encoder.layer)
        self.top = layers[len(layers) - k:] if k else []
        for layer in self.top:
            for p in layer.parameters():
                p.requires_grad = True
        self.norm = nn.LayerNorm(d, elementwise_affine=False)
        self.task_query = nn.Parameter(torch.empty(d).uniform_(-.05, .05)) if flags[0] else None
        self.cross_query = nn.Linear(14, d) if flags[1] else None
        self.gate = nn.Linear(15, 1) if flags[2] else None
        self.head = nn.Sequential(nn.Linear(14 + 3 * d + 2, 64), nn.ReLU(), nn.Dropout(DROPOUT),
                                  nn.Linear(64, 32), nn.ReLU(), nn.Dropout(DROPOUT), nn.Linear(32, 1))

    def heads(self):
        mods = [self.head, self.cross_query, self.gate, self.norm]
        return [p for m in mods if m is not None for p in m.parameters()] + ([self.task_query] if self.task_query is not None else [])

    def set_mode(self, train):
        self.eval()
        if train:
            for m in [self.head] + self.top:
                m.train()

    def forward(self, input_ids, mask, day_idx, n_days, x):
        market, avail = x[:, :14], x[:, 14:]
        h = self.enc(input_ids=input_ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(h.dtype)
        art = self.norm(((h * m).sum(1) / m.sum(1)).float())                    # (N, d) article embeddings
        counts = torch.bincount(day_idx, minlength=n_days)
        pos = torch.arange(len(day_idx), device=art.device) - (counts.cumsum(0) - counts)[day_idx]
        width = int(counts.max())
        padded = torch.zeros(n_days, width, self.d, device=art.device).index_put((day_idx, pos), art)
        pad = torch.arange(width, device=art.device).unsqueeze(0) >= counts.unsqueeze(1)   # True = padding
        present = counts > 0
        mean = padded.sum(1) / counts.clamp_min(1).unsqueeze(1)
        task, cross = torch.zeros_like(mean), torch.zeros_like(mean)
        for name, active in [("task", self.flags[0]), ("cross", self.flags[1])]:
            if not active:
                continue
            q = self.task_query.expand(n_days, -1) if name == "task" else self.cross_query(market)
            scores = (padded * q.unsqueeze(1)).sum(-1) / self.d ** .5
            w = torch.softmax(scores.masked_fill(pad, -1e9), dim=1) * (~pad)
            s = (w.unsqueeze(-1) * padded).sum(1) * present.unsqueeze(1)
            task, cross = (s, cross) if name == "task" else (task, s)
        g = torch.sigmoid(self.gate(torch.cat([market, avail[:, :1]], 1))) if self.gate is not None else torch.ones_like(avail[:, :1])
        text = torch.cat([mean, task, cross], 1) * g * present.unsqueeze(1)
        return self.head(torch.cat([market, text, avail], 1)).squeeze(-1)


def collate(rows, X, dv):
    seqs, day_idx = [], []
    for j, r in enumerate(rows):
        for a in day_articles[r]:
            seqs.append(enc_ids[a]); day_idx.append(j)
    L = max(len(s) for s in seqs)
    input_ids = torch.zeros(len(seqs), L, dtype=torch.long)
    mask = torch.zeros(len(seqs), L, dtype=torch.long)
    for i, s in enumerate(seqs):
        input_ids[i, :len(s)] = torch.tensor(s); mask[i, :len(s)] = 1
    return (input_ids.to(dv), mask.to(dv), torch.tensor(day_idx, device=dv), len(rows),
            torch.tensor(X[rows], dtype=torch.float32, device=dv))


@torch.no_grad()
def predict(model, rows, X, dv, batch=16):
    model.set_mode(False)
    out = []
    for s in range(0, len(rows), batch):
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            out.append(torch.sigmoid(model(*collate(rows[s:s + batch], X, dv))).float().cpu().numpy())
    return np.concatenate(out)


def score(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return {"log_loss": log_loss(y, p, labels=[0, 1]), "auc": roc_auc_score(y, p),
            "accuracy": accuracy_score(y, p >= 0.5), "mcc": matthews_corrcoef(y, p >= 0.5), "pred_up": float((p >= 0.5).mean())}


def train_model(k, seed, tr, X, y, dv, epochs, on_epoch=None):
    rng = np.random.RandomState(seed)
    with LOCK:
        torch.manual_seed(seed)
        model = A111(k).to(dv)
    enc_params = [p for p in model.enc.parameters() if p.requires_grad]
    groups = [{"params": model.heads(), "lr": LR_HEAD}] + ([{"params": enc_params, "lr": LR_ENC}] if enc_params else [])
    opt = torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)
    gs = torch.cuda.amp.GradScaler(enabled=device == "cuda")
    for epoch in range(1, epochs + 1):
        model.set_mode(True)
        order = rng.permutation(tr)
        for s in range(0, len(order), BATCH_DAYS):
            b = order[s:s + BATCH_DAYS]
            with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                logit = model(*collate(b, X, dv))
            loss = nn.functional.binary_cross_entropy_with_logits(logit.float(), torch.tensor(y[b], dtype=torch.float32, device=dv))
            opt.zero_grad(); gs.scale(loss).backward(); gs.step(opt); gs.update()
        if on_epoch:
            on_epoch(epoch, model)
    return model


def fit(k, seed, tr, va, X, y, dv):
    rows = []
    model = train_model(k, seed, tr, X, y, dv, EPOCHS, lambda e, m: rows.append({"epoch": e, **score(y[va], predict(m, va, X, dv))}))
    del model; torch.cuda.empty_cache()
    return pd.DataFrame(rows)

In [ ]:
y = dev.y.to_numpy(int)
history, baselines = [], []
t0 = time.time()
for fold in ["1", "2", "3"]:
    role = dev.fold_roles.map(lambda r: r[fold])
    tr, va = np.where(role == "train")[0], np.where(role == "val")[0]
    if QUICK_TEST_DAYS:
        tr, va = tr[-QUICK_TEST_DAYS:], va[:QUICK_TEST_DAYS]
    X = make_X(dev, tr)
    lr = LogisticRegression(C=0.1, max_iter=1000).fit(X[tr, :14], y[tr])
    baselines += [{"fold": fold, "method": "train_prior", **score(y[va], np.full(len(va), y[tr].mean()))},
                  {"fold": fold, "method": "price_only_logreg", **score(y[va], lr.predict_proba(X[va, :14])[:, 1])}]

    def job(k, seed, fold=fold, tr=tr, va=va, X=X):
        dv = GPU_POOL.get()
        try:
            h = fit(k, seed, tr, va, X, y, dv)
        finally:
            GPU_POOL.put(dv)
        h.insert(0, "fold", fold); h.insert(1, "k", k); h.insert(2, "seed", seed)
        b = h.loc[h.log_loss.idxmin()]
        print(f"fold {fold} K={k} seed {seed} [{dv}]: best ep {int(b.epoch)} ll {b.log_loss:.4f} auc {b.auc:.3f} | "
              f"last ll {h.log_loss.iloc[-1]:.4f} | {time.time() - t0:.0f}s", flush=True)
        return h
    with ThreadPoolExecutor(N_GPU) as ex:
        futs = [ex.submit(job, k, seed) for k in UNFREEZE for seed in SEEDS]
        history += [f.result() for f in futs]
    pd.concat(history).to_csv(OUT_DIR / "history.csv", index=False)
pd.DataFrame(baselines).to_csv(OUT_DIR / "baselines.csv", index=False)

In [ ]:
history_df = pd.concat(history)
best = history_df.loc[history_df.groupby(["fold", "k", "seed"]).log_loss.idxmin()]
last = history_df[history_df.epoch == EPOCHS]
cols = ["log_loss", "auc", "accuracy", "mcc", "pred_up"]
summary = pd.concat({"best epoch (val log loss)": best.groupby("k")[cols].mean(),
                     "last epoch": last.groupby("k")[cols].mean()}).round(4)
base = pd.DataFrame(baselines).groupby("method")[cols].mean().round(4)
print("A111 + FinBERT (mean over folds x seeds)"); display(summary)
print("Baselines (mean over folds)"); display(base)
print("Per fold, best epoch"); display(best.groupby(["fold", "k"])[cols].mean().round(4))
summary.to_csv(OUT_DIR / "summary.csv")

## How to read this

- Interesting only if `K=2` beats `K=0` **and** both baselines on validation log loss in at least 2 of 3 folds. Watch `pred_up`: values near 100% mean the model collapsed to the majority class.
- Compare best-epoch with last-epoch rows to see selection optimism.

## Final test evaluation (run once)

Set the frozen config from validation, then run the next two cells once. Refit on all 930 train+val rows with fixed epochs, 3 seeds, predict test once. Run `K=0` and `K=2` as a pre-specified pair, nothing else. The test period was already used by earlier experiments, so report this as exploratory.

In [ ]:
RUN_TEST = False             # True only after the config below is frozen
FINAL_K = 2
FINAL_EPOCHS = 2             # median best epoch for FINAL_K (printed below)
print(history_df[history_df.k == FINAL_K].loc[lambda d: d.groupby(["fold", "seed"]).log_loss.idxmin()].epoch.describe()[["mean", "50%"]])

In [ ]:
assert RUN_TEST, "Set RUN_TEST = True after freezing FINAL_K / FINAL_EPOCHS"
te = load(["test.csv"])
assert te.Date.min() > dev.Date.max()
n_dev = len(dev)
register(te)
full = pd.concat([dev, te], ignore_index=True)
yf = full.y.to_numpy(int)
tr_all, te_idx = np.arange(n_dev), np.arange(n_dev, len(full))
Xf = make_X(full, tr_all)

def final_job(s):
    dv = GPU_POOL.get()
    try:
        model = train_model(FINAL_K, s, tr_all, Xf, yf, dv, FINAL_EPOCHS)
        p = predict(model, te_idx, Xf, dv)          # test touched once, after training ends
        del model; torch.cuda.empty_cache()
        return p
    finally:
        GPU_POOL.put(dv)
with ThreadPoolExecutor(N_GPU) as ex:
    probs = dict(zip(SEEDS, ex.map(final_job, SEEDS)))
np.save(OUT_DIR / f"test_probs_K{FINAL_K}.npy", np.stack(list(probs.values())))

In [ ]:
yt = yf[te_idx]
lr = LogisticRegression(C=0.1, max_iter=1000).fit(Xf[tr_all, :14], yf[tr_all])
streams = {
    f"a111_finbert_K{FINAL_K}": list(probs.values()),
    "train_prior": [np.full(len(yt), yf[tr_all].mean())],
    "price_only_logreg": [lr.predict_proba(Xf[te_idx, :14])[:, 1]],
    "last_direction": [(full.loc[te_idx, "log_return_t"].to_numpy() > 0).astype(float).clip(1e-6, 1 - 1e-6)],
}
res = pd.DataFrame([{"method": m, **pd.DataFrame([score(yt, p) for p in ps]).mean().to_dict()} for m, ps in streams.items()]).round(4)
display(res); res.to_csv(OUT_DIR / f"test_results_K{FINAL_K}.csv", index=False)

def boot(p, blocks=20, B=1000, seed=0):
    r = np.random.RandomState(seed); n = len(yt); out = []
    for _ in range(B):
        idx = np.concatenate([(s + np.arange(blocks)) % n for s in r.randint(0, n, int(np.ceil(n / blocks)))])[:n]
        if len(set(yt[idx])) < 2: continue
        out.append((matthews_corrcoef(yt[idx], p[idx] >= .5), roc_auc_score(yt[idx], p[idx])))
    return np.percentile(np.array(out), [2.5, 97.5], axis=0)
pm = np.mean(list(probs.values()), axis=0)
ci = boot(pm)
print(f"ensemble MCC 95% CI [{ci[0,0]:.3f}, {ci[1,0]:.3f}] | AUC 95% CI [{ci[0,1]:.3f}, {ci[1,1]:.3f}]")
print("Majority-class accuracy (always UP):", max(yt.mean(), 1 - yt.mean()).round(4), "| Test up-rate:", yt.mean().round(3), "| model predicted up:", (pm >= .5).mean().round(3))